# Households

Household totals by county and growth from base year to targets year — and the implied delta vs. the input city-level targets.

In [1]:
import sys
from pathlib import Path

# Make summary_scripts/ importable regardless of where the kernel started.
_HERE = Path.cwd()
for candidate in [_HERE, _HERE / "summary_scripts", _HERE.parent / "summary_scripts"]:
    if (candidate / "validation_data_input.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from validation_data_input import (
    load_config, load_settings, load_unrolled, load_unrolled_regional,
    load_targets, control_id_lookup,
    load_legacy_pop, load_legacy_units, load_legacy_emp,
    available_years,
)
from util import (
    COUNTY_ID_TO_NAME, COUNTY_ORDER, COUNTY_COLORS, INDICATORS,
    apply_plotly_theme, format_int, pct_diff, style_diff_table, passfail_badge,
)

cfg = load_config()
settings = load_settings()
BASE_YEAR = settings.get("base_year")
END_YEAR = settings.get("end_year")
TARGETS_END_YEAR = settings.get("targets_end_year")

ctrl = load_unrolled().merge(control_id_lookup(), on="control_id", how="left")
ctrl["county"] = ctrl["county_id"].map(COUNTY_ID_TO_NAME)


## Households by county

In [2]:
hh = ctrl.groupby(['county', 'year'])['total_hh'].sum().unstack('year')
hh = hh.reindex(COUNTY_ORDER)
summary = pd.DataFrame({
    f'HH {BASE_YEAR}': hh[BASE_YEAR],
    f'HH {TARGETS_END_YEAR}': hh[TARGETS_END_YEAR],
    'Growth': hh[TARGETS_END_YEAR] - hh[BASE_YEAR],
    '% Growth': (hh[TARGETS_END_YEAR] / hh[BASE_YEAR] - 1) * 100,
})
summary.loc['Region'] = summary.sum(numeric_only=True)
summary.loc['Region', '% Growth'] = (
    summary.loc['Region', f'HH {TARGETS_END_YEAR}']
    / summary.loc['Region', f'HH {BASE_YEAR}'] - 1
) * 100
style_diff_table(
    summary,
    int_cols=[f'HH {BASE_YEAR}', f'HH {TARGETS_END_YEAR}', 'Growth'],
    pct_cols=['% Growth'], pct_threshold=100,
)

,HH 2023,HH 2044,Growth,% Growth
county,,,,
King,"952,893","1,141,705","188,812",+19.8%
Kitsap,"106,813","130,688","23,875",+22.4%
Pierce,"345,236","433,147","87,911",+25.5%
Snohomish,"318,194","459,469","141,275",+44.4%
Region,"1,723,136","2,165,009","441,873",+25.6%


## Growth (base → targets) by county

In [3]:
growth = (hh[TARGETS_END_YEAR] - hh[BASE_YEAR]).reset_index()
growth.columns = ['county', 'growth']
fig = px.bar(
    growth, x='county', y='growth', color='county',
    category_orders={'county': COUNTY_ORDER},
    color_discrete_map=COUNTY_COLORS,
    title=f'Household growth, {BASE_YEAR}→{TARGETS_END_YEAR}',
    labels={'growth': 'Households added', 'county': 'County'},
)
fig.update_layout(showlegend=False)
apply_plotly_theme(fig)

## Output vs. input city targets (CityHH)

In [4]:
targets = load_targets('CityHH')
# CityHH columns: RGID, county_id, control_id, Juris, HH<base_year>, HH<base_year>+5..., HHGro<base..targets>, HH<end_year>
base_col = f'HH{BASE_YEAR}'
# growth column name is e.g. HHGro2350 when base=2023, targets_end=2050
gro_col = next(c for c in targets.columns if c.startswith('HHGro'))
ctrl_end = (
    ctrl.loc[ctrl['year'] == TARGETS_END_YEAR, ['control_id', 'total_hh']]
    .rename(columns={'total_hh': 'HH_end_ctrl'})
)
compare = targets.merge(ctrl_end, on='control_id', how='left')
compare['county'] = compare['county_id'].map(COUNTY_ID_TO_NAME)
by_co = compare.groupby('county').agg(
    target=(gro_col, 'sum'),
    base=(base_col, 'sum'),
    end=('HH_end_ctrl', 'sum'),
)
by_co = by_co.reindex(COUNTY_ORDER)
by_co['ctrl_growth'] = by_co['end'] - by_co['base']
by_co['delta'] = by_co['ctrl_growth'] - by_co['target']
by_co['% diff vs. target'] = by_co.apply(
    lambda r: pct_diff(r['ctrl_growth'], r['target']), axis=1
)
by_co = by_co.rename(columns={
    'target': 'Target growth',
    'ctrl_growth': 'Controls growth',
    'delta': 'Controls − Target',
})[['Target growth', 'Controls growth', 'Controls − Target', '% diff vs. target']]
style_diff_table(
    by_co,
    int_cols=['Target growth', 'Controls growth', 'Controls − Target'],
    pct_cols=['% diff vs. target'], pct_threshold=2,
)

,Target growth,Controls growth,Controls − Target,% diff vs. target
county,,,,
King,"242,764","188,813","-53,951",-22.2%
Kitsap,"30,702","23,878","-6,824",-22.2%
Pierce,"113,033","87,914","-25,119",-22.2%
Snohomish,"181,640","141,276","-40,364",-22.2%


## Regional households — controls vs. legacy LUVit

In [5]:
reg = load_unrolled_regional()[['year', 'total_hh']].copy()
reg['source'] = 'New pipeline'
legacy = load_legacy_units()
frames = [reg]
if legacy is not None and 'total_hh' in legacy.columns:
    leg = legacy[['year', 'total_hh']].copy()
    leg['source'] = 'Legacy LUVit'
    frames.append(leg)
compare = pd.concat(frames, ignore_index=True)
fig = px.line(
    compare, x='year', y='total_hh', color='source', markers=True,
    title='Regional households — pipeline vs. legacy',
    labels={'total_hh': 'Households', 'year': 'Year', 'source': 'Source'},
)
if legacy is None:
    fig.add_annotation(
        text='Legacy LUVit estimates not available for this run.',
        xref='paper', yref='paper', x=0.5, y=1.08,
        showarrow=False, font=dict(color='#888'),
    )
apply_plotly_theme(fig)

## Notes

- The growth column in `CityHH` is named `HHGro<base><targets>` (e.g. `HHGro2350` for base 2023 and targets 2050) and is sourced from `TargetsRebasedOutput.xlsx`.
- A small percent difference is expected from rounding and from any RG-level scaling applied during interpolation.
- Legacy comparison reads the most recent `LUVit_ct_by_tod_generator-*.xlsx` under the `legacy_path` configured in `validation/config.yaml`.